# Lab 3 — Getting the time axis right
### EP0418 · Topic 3 · 2 practical hours

**You need:** `ep0418.py` and `sg_taxi_hourly.csv` beside this notebook.

---
This lab uses real Singapore taxi availability pulled from `data.gov.sg`. It is a
live government feed, and it does something that will break your analysis if you
do not notice it.

## Putting the assistant's code into a TODO cell

A TODO cell has a line ending in `...`. Replace **only that line** with the
assistant's code, and keep the name on the left (`s`, `clean`, `H`). Delete any
`import` or `read_csv` lines it adds; the data is already loaded. If it used a
different name for the result, add one line at the end to rename it, e.g.
`s = hourly_df`.

---
## How every lab in this module works

You will **not** write this code from scratch. Ask your AI assistant to write it.
Your job is the part it cannot do: deciding what the right answer looks like,
reading what it gave you, and proving it did what you asked.

Every prompt in these notebooks opens with the rules that matter for that task:
do not drop rows, show the row count before and after, treat -999 as missing,
explain the code in plain words. Keep those lines when you adapt a prompt, and
start your own prompts the same way.

At the foot of each task is a **verification contract** — a few `check.*` lines.
**You may not edit them.** The AI can write any implementation it likes; it cannot
satisfy these by accident.

If a check fails, read the message. It tells you what was found and what to try
next. Failing checks are the normal state of a lab in progress, not a sign you
are behind.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check, show


def filled(*values):
    """True once every TODO slot above has been replaced. Otherwise says so."""
    if any(v is ... or v is None for v in values):
        print("A TODO above is still '...'. Fill it in, then run this cell again.")
        return False
    return True


def submit_prediction(team, lab, prediction, method):
    """Send one team's number to the Prediction League board."""
    import urllib.parse
    form = "https://docs.google.com/forms/d/e/1FAIpQLSdwpzHafC_rmPDdbK_cm6J8CihD-NSVS7ZBUyYCWIrS_MMJ0g/formResponse"
    fields = {'team': 'entry.911171418', 'lab': 'entry.1890998159', 'prediction': 'entry.959891957', 'method': 'entry.1997288724'}
    if not team or not method or prediction is None:
        print("Fill in TEAM, MY_PREDICTION and METHOD above, then run this cell again."); return
    entry = {"team": team, "lab": str(lab), "prediction": f"{float(prediction):.1f}", "method": method}
    if not form:
        print("The Prediction League is not switched on yet; tell your tutor your number."); return
    data = urllib.parse.urlencode({fields[k]: v for k, v in entry.items()}).encode()
    try:
        urllib.request.urlopen(urllib.request.Request(form, data=data, headers={"User-Agent": "EP0418"}), timeout=60)
        print(f"Entered: {team} predicts {float(prediction):.1f}. Check the scoreboard in a minute.")
    except Exception as e:
        print("The league did not answer:", e, "- try again in a minute.")

CSV = "sg_taxi_hourly.csv"
fetch(CSV)

raw = pd.read_csv(CSV, parse_dates=["timestamp"])
print(f"{len(raw):,} rows")
raw.head()

## Task 1 — Look at the timestamps before anything else

Run the cell below and read the timestamps carefully.

In [ ]:
print(raw.timestamp.head(6).to_string(index=False))
print()
print("gaps between consecutive readings:")
print(raw.timestamp.diff().value_counts().head(5).to_string())

Now the picture. **Look for:** the daily rhythm, and whether you can see where
readings are missing.

In [ ]:
show.series(raw, col="taxi_count", title="Taxis available, every reading, eight weeks")
hours_since_previous = raw.timestamp.diff().dt.total_seconds() / 3600
show.series(pd.Series(hours_since_previous.values, index=raw.timestamp),
            title="Hours since the previous reading", ylabel="hours", hline=1)

**What you should see:** in the first chart, a strong daily cycle and a weekly one
on top of it, and nothing that looks wrong. That is the trap: the line is drawn
straight across any gap, so missing hours are invisible. The second chart is the
honest one: most readings are one hour apart, and the spikes are the hours the
feed missed, some of them several hours long. A chart of the values cannot show
you what is not there. A chart of the gaps can.

### They are not on the hour

`23:59:48`, then `00:59:50`, then `01:59:53`. The API returns the moment the
reading was actually taken, not a tidy clock tick.

This is **irregular sampling**, and here is why it matters immediately:

If you want "the same hour last week" and you write `.shift(168)`, you are
shifting by **168 rows** — and rows are not evenly spaced in time. You will land
somewhere near last week, off by an unpredictable amount, and your comparison
will be quietly meaningless.

Fix the grid first. Then shift.

## Task 2 — Put it on a regular hourly grid

Prompt your assistant with something like:

> I have a DataFrame `raw` with a `timestamp` column and a `taxi_count` column.
> The timestamps are irregular. Resample onto a regular hourly grid, taking the
> mean within each hour. Do not drop empty hours — leave them as NaN. Then show me
> the row count before and after, and how many hours are empty.

Note the constraint in that last sentence. Without it you will very likely get
code that quietly discards the gaps.

In [ ]:
# TODO(you): make `s` a regular hourly Series of taxi_count
s = ...

# ---- verification contract — do not edit ----
check.regular_cadence(s, "1h")
check.report()

**Expected output** *(computed from the data file when this notebook was built)*: `s` has 1,344 hourly slots, one per hour of the eight weeks. If you have more, the grid is finer than hourly; if fewer, the gaps were dropped.

**Look for** the difference from the first chart in Task 1.

In [ ]:
if filled(s):
    show.series(s, title="On a regular hourly grid: the line now breaks where hours are empty")

**What you should see:** the same shape, now with small breaks in the line. Each
break is an empty hour that the grid has made visible. If your chart has no
breaks, your assistant filled the gaps with something, probably zeros, and the
check in Task 3 will tell you how many hours that affected.

## Task 3 — How many hours are actually missing?

Resampling does not invent data. It creates the empty slots so you can *see* what
was never there.

> I have a pandas Series `s` with an hourly DatetimeIndex. Print how many values
> are NaN and what fraction of the total that is. Do not change `s`.

In [ ]:
# TODO(you): print how many slots in `s` are NaN, and what fraction that is
...

**Expected output** *(computed from the data file when this notebook was built)*: 39 empty slots, 2.9% of the total. Zero empty slots means they were filled with something; find out with what.

## Task 4 — Now the shift means what you think it means

With a regular grid, `shift(168)` really is one week (7 days × 24 hours).

**MAE** (mean absolute error) is the average size of the miss: for each hour,
how far the guess was from the truth, ignoring whether it was too high or too
low, averaged over all hours.

> `s` is an hourly pandas Series. Build `snaive`, which is `s` shifted by 168
> hours. Then compute the mean absolute error between `s` and `snaive`, using
> only rows where both exist, and call it `mae_snaive`. Say how many rows were used.

In [ ]:
# TODO(you): build `snaive` = the value from 168 hours earlier
snaive = ...

# TODO(you): compute the mean absolute error between s and snaive,
#            using only rows where BOTH exist
mae_snaive = ...

if filled(snaive, mae_snaive):
    print(f"seasonal-naive MAE: {mae_snaive:,.1f}")
    print(f"mean taxi count   : {s.mean():,.0f}")

# ---- verification contract — do not edit ----
# The baseline's MAE is compared with the series' standard deviation, which is
# roughly the error of always guessing the average. The report calls the first
# number "Model" because the check is general; here it is your baseline.
if filled(s):
    check.compare_to_baseline(mae_snaive, s.std(), metric="MAE")
check.report()

**Expected output** *(computed from the data file when this notebook was built)*: seasonal-naive MAE about 496 taxis, against a mean count of about 2,375. A number near 713 or above usually means the two series were not aligned before subtracting.

**Look for** how close "the same hour last week" tracks the real line, and where
it misses.

In [ ]:
if filled(snaive):
    show.compare(s.iloc[-24 * 14:], same_hour_last_week=snaive.iloc[-24 * 14:],
                 title="Last two weeks: actual against the seasonal-naive guess")

**What you should see:** the dashed guess follows the real line closely most of
the time, because the daily cycle repeats. Where they part is where something
unusual happened in one of the two weeks: that is the error the MAE is averaging,
and it is what a better model would have to explain.

## Task 5 — See the two cycles

The helper draws the two profiles. **Look for** the lowest and highest hours, and
which days differ from the rest.

In [ ]:
if filled(s):
    show.profile(s, by="hour", title="Mean taxis available by hour of day")
    show.profile(s, by="dow", title="Mean taxis available by day of week")

**What you should see:** availability lowest at about three in the morning and
highest in the evening, and the two weekend days sitting well below the weekdays. Write down the hour at which availability is lowest; the sheet asks for
it. Then ask your assistant to draw the same two charts without the helper, and
check its version against this one.

---
## Going further (optional): a real market's time axis

For the adventurous. Real hourly share prices have their own time-axis problems.
This uses **yfinance**, a free, unofficial library for Yahoo Finance data, meant for
personal, educational use (pypi.org/project/yfinance).

Set `TRY_LIVE = True` and run the cell. It downloads a month of hourly prices for
ES3.SI (an ETF of Singapore shares) once, saves them, and reuses the saved copy, so
the class does not hit Yahoo's limits. If Yahoo refuses, try again later.

Then answer:
1. How many rows does each trading day have? Where are the gaps?
2. What time zone are the timestamps in?
3. Should you put this onto a regular hourly timetable, as you did with the taxi
   feed? Why not?

In [ ]:
TRY_LIVE = False      # set to True to try it

if TRY_LIVE:
    import os
    import yfinance as yf          # if this fails: run  !pip install yfinance  first
    if not os.path.exists("es3_hourly.csv"):
        yf.Ticker("ES3.SI").history(period="1mo", interval="1h").to_csv("es3_hourly.csv")
    live = pd.read_csv("es3_hourly.csv", index_col=0)
    live.index = pd.to_datetime(live.index, utc=True).tz_convert("Asia/Singapore")
    print("time zone:", live.index.tz)
    print(live.groupby(live.index.date).size().tail(10))   # rows per trading day

**My answers:**

1.
2.
3.

---
## Predict: How many taxis at 8 am on Monday? (10 minutes, teams of two or three)

*The Prediction League, week 2. Same team name as last time.*

How many taxis will be available for hire across Singapore at 8:00 am on the Monday of the target week? You have the hourly history; Lab 3 gave you the regular grid, shift() and 'the same hour last week'. The baseline, doing nothing clever, is on the board already:
the average of every Monday 8 am in the file. The target week's dates are on the scoreboard, and last week's
result is there too: look at who was closest and what they said they did.

In [ ]:
fetch("sg_taxi_hourly.csv")
taxi = pd.read_csv("sg_taxi_hourly.csv", parse_dates=["timestamp"])
taxi["timestamp"] = taxi.timestamp.dt.tz_localize(None)
mon8 = taxi[(taxi.timestamp.dt.weekday == 0) & (taxi.timestamp.dt.hour == 8)]
print(f"every Monday 8 am in the file : {mon8.taxi_count.round().astype(int).tolist()}")
print(f"their average                 : {mon8.taxi_count.mean():6.0f} taxis   <- the baseline")
print(f"the most recent Monday 8 am   : {mon8.taxi_count.iloc[-1]:6.0f} taxis   (the same hour last week)")

**Step by step.** Open Gemini (spark icon, bottom right), **+** for a new chat,
paste, Enter:

> I am new to this. Answer in three plain sentences, no headings, no code.
> How many taxis will be available for hire across Singapore at 8:00 am on the Monday of the target week? What would you base the guess on, using what this week's lab taught,
> and why might it still be wrong?

Read the three sentences. Decide with your team: the baseline as printed, one of
the other numbers printed above, or a number Gemini computes for you (rules first,
**+ Code**, paste, run, read). Fill in the three lines below, for example
`TEAM = "Team Rocket"`, `MY_PREDICTION = 2350`, `METHOD = "the baseline, nudged
for ..."`, and run the cell. Ignore Gemini's blue suggestion chips.

In [ ]:
TEAM = ""                 # TODO(you): the same team name as last week
MY_PREDICTION = None      # TODO(you): your number, e.g. 2350
METHOD = ""               # TODO(you): one line on how you got it

submit_prediction(TEAM, 3, MY_PREDICTION, METHOD)

---
## Read before you run

An assistant was asked to put the taxi feed onto a regular hourly grid and “make sure there are no missing values”. It wrote:

The code is in the cell below, commented out. It came from an AI assistant in
someone else's chat; you are not asking Gemini for it. **Do not run it yet.** Read
it, answer the two questions below, and only then remove the `#` signs and run it
to see whether you were right.

What does the second line do to the 39 empty hours, and why does that matter for anything computed from `s` afterwards?

Write down what happens to the minimum and the mean of `s`.

<details><summary>Hint 1: where to look</summary>

What value does `fillna(0)` write into an empty hour? Is zero a measurement of taxis, or the absence of one?

</details>
<details><summary>Hint 2: closer to the answer</summary>

Look at `s.min()` and `s.mean()` before and after the second line. The 39 empty hours become hours with 0 taxis, and every check still passes.

</details>

**After you have written your answer, and not before:** select the cell, click the
pencil-and-spark icon at its top right and choose **Explain code**. Did Gemini notice
the fault? It often does not. That is the point: an explanation that reads well is
not evidence the code is right.

**To fix it:** choose **Transform code** on the same cell and say what to change, in
plain words. Colab shows the change as a diff before it is applied. Read the diff. If
it touches a `check.` line, reject it.

In [ ]:
# An AI assistant wrote this for a classmate. It is already here: there is nothing to ask Gemini for.
# 1. Read it. Answer the two questions in the cell below, in your own words.
# 2. Then select this cell, click the pencil-and-spark icon, choose Explain code. Did Gemini see the fault?
# 3. Only then remove the # signs and run it, to see whether you were right.
# s = raw.set_index("timestamp").taxi_count.resample("h").mean()
# s = s.fillna(0)

**What is wrong with it:**

**What it would produce, and what it should produce:**

---
## Quiz yourself before you leave (5 minutes)

The checks tell you whether the code is right. They do not tell you whether
**you** know what it did. Paste this into the Gemini chat:

> Quiz me on this lab. Ask five short questions, one at a time, in plain words, on these ideas: why the taxi timestamps are not on the hour; what resampling onto a regular grid does to the gaps; why shift(168) only means one week on a regular grid; what a seasonal-naive forecast is; what mean absolute error measures. Take the examples from the code and outputs in this notebook. Wait for my answer. Then say in one sentence whether I am right, and why. Do not give the next question until I answer. After the fifth, give my score out of 5 and the one idea I should revise, in one line.

Gemini asks in the chat panel; you answer in the chat box; it marks you and asks
the next. Ignore the suggestion chips under its messages. Answer in your own
words, without looking back until it has marked you.

Then write your score and the idea to revise below. Your tutor asks for them in
the plenary.

**Quiz score:** ___ / 5

**The idea I should revise, in my own words:**

---
## Before you leave

**Optional: the Trading League opens today.** `shift()` on a regular grid is
exactly what a trading rule is built from: yesterday's close, last week's close, a
moving average. If you want a game to play with it, open the League notebook from
the scoreboard page (https://stanleysp-eng.github.io/EP0418-labs/), do Tasks 1 to
3, and your entry is on the board. Not graded, not required, open until the end of
Week 7.

**Mistake caught** (below):

- What did your assistant produce for Task 2 on the first attempt? Did it drop the
  empty hours, or keep them?
- How many hours were missing from the feed, and did that surprise you?

**One thing to notice:** the minimum in this data is about 24 taxis available
across all of Singapore. That is not plausible — it is a glitch in the feed. It is
real, it is not something we added, and you will have to decide what to do with it
in your project.

**Mistake caught:** *(what I expected · what was there · how I found it)*